# 17 — Classification Susceptibility Analysis

This notebook investigates what determines an EEG subject's **classification susceptibility**.

Here, classification susceptibility is defined as the subject-level within-subject LDA ROC-AUC from the grouped benchmark. The goal is to correlate this LDA performance with subject-level EEG features computed from the raw/minimally processed epoch data.

Research question:

> Find out what separates a subject with high classification accuracy from low classification accuracy. Use within-subject LDA only for now, and correlate the LDA classifier accuracy across subjects with other features computed on the raw data, such as power and SNR. Correlate within datasets and also across z-scored datasets.

The notebook therefore computes:

1. Subject-level LDA AUC from saved benchmark results.
2. Subject-level EEG features:
   - ERP target-standard difference strength
   - ERP SNR
   - baseline noise
   - trial counts and class balance
   - spectral bandpower features
3. Within-dataset correlations.
4. Grand cross-dataset correlations after z-scoring within each dataset.

In [ ]:
# Run from the repository root so that relative result paths resolve
# identically regardless of where this notebook lives.
from oddball_eeg.paths import use_project_root

PROJECT_ROOT = use_project_root()

In [ ]:
from __future__ import annotations

from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne

from scipy.signal import welch
from scipy.stats import pearsonr, spearmanr

from oddball_eeg.bids import make_epoching_config_for_dataset
from oddball_eeg.config import get_experiment
from oddball_eeg.io import save_dataframe
from oddball_eeg.loader import load_one_subject_grouped
from oddball_eeg.stats import strength_word, zscore_within_group
from oddball_eeg.visualization import (
    round_numeric_columns,
    save_figure,
    set_pandas_display_options,
    valid_topomap_picks,
)


set_pandas_display_options()
mne.set_log_level("ERROR")

# ------------------------------------------------------------
# Main pooled datasets for susceptibility analysis
# ------------------------------------------------------------
# Kept focused on the research question:
# within-subject LDA only, across pooled datasets.
# ------------------------------------------------------------

EXPERIMENT_NAMES = [
    "ds003061_within_subject_max30_wide_timebin",
    "ds005863_within_subject_max30_wide_timebin",
    "ds006466_within_subject_max30_wide_timebin",
    "ds006480_within_subject_max30_wide_timebin",
]

# Optional later extension, not used in the main grand correlation
# to avoid duplicating ds006480 subjects in active/passive forms.
DS006480_CONDITION_EXPERIMENT_NAMES = [
    # "ds006480_active_within_subject_max15",
    # "ds006480_passive_within_subject_max15",
]

MODEL_NAME = "lda"

# ------------------------------------------------------------
# Feature extraction settings
# ------------------------------------------------------------

# Use a broad raw-like signal range for susceptibility features.
# This is not the LDA model input. It is for subject-level feature analysis.
FEATURE_L_FREQ = 0.1
FEATURE_H_FREQ = 45.0
FEATURE_NOTCH = 60.0
FEATURE_RESAMPLE_SFREQ = 128.0

EPOCH_TMIN = -0.2
EPOCH_TMAX = 0.8
BASELINE_WINDOW = (-0.2, 0.0)
P300_WINDOW = (0.25, 0.50)

ROI_CANDIDATES = ["Pz", "P3", "P4", "CPz", "Cz", "CP1", "CP2"]

# Spectral bands for raw/minimally processed epoch features
BANDS = {
    "delta_1_4": (1.0, 4.0),
    "theta_4_8": (4.0, 8.0),
    "alpha_8_12": (8.0, 12.0),
    "beta_13_30": (13.0, 30.0),
    "gamma_30_45": (30.0, 45.0),
    "low_1_12": (1.0, 12.0),
    "high_30_45": (30.0, 45.0),
}

# ------------------------------------------------------------
# Output folders
# ------------------------------------------------------------

OUT_ROOT = Path("results") / "thesis_artifacts" / "classification_susceptibility"
TABLE_DIR = OUT_ROOT / "tables"
FIG_DIR = OUT_ROOT / "figures"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("Output root:", OUT_ROOT)
print("Experiments:")
for name in EXPERIMENT_NAMES:
    print(" -", name)

In [ ]:
def normalize_subject_for_cfg(subject_value, cfg) -> str:
    """
    Normalize subject IDs so saved CSV values such as 1 match config subjects such as '001' or '01'.
    """
    s = str(subject_value).strip()

    if s.endswith(".0"):
        s = s[:-2]

    if not s.isdigit():
        return s

    configured = [str(x) for x in cfg.subjects.subjects]
    digit_lengths = [len(x) for x in configured if str(x).isdigit()]

    if len(digit_lengths) == 0:
        return s

    width = max(digit_lengths)
    return s.zfill(width)


def dataset_label_from_cfg(cfg) -> str:
    """
    Human-readable dataset/condition label used for grouping.
    """
    ds = cfg.dataset.dataset
    task = cfg.dataset.task
    preset = cfg.dataset.event_preset

    if ds == "ds006480":
        return f"{ds}_{preset}"

    return ds


def load_subject_level_lda_auc(exp_name: str) -> pd.DataFrame:
    """
    Load subject-level within-subject LDA AUC from the saved benchmark result table.
    """
    cfg = get_experiment(exp_name)

    summary_path = cfg.table_dir("summaries") / "subject_summary_across_seeds.csv"

    if not summary_path.exists():
        raise FileNotFoundError(
            f"Missing subject summary table for {exp_name}:\n{summary_path}"
        )

    df = pd.read_csv(summary_path)

    required = {"subject", "model", "auc_seed_mean"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(
            f"{summary_path} is missing required columns: {sorted(missing)}"
        )

    df["model"] = df["model"].astype(str).str.lower().str.strip()
    df = df[df["model"] == MODEL_NAME].copy()

    if df.empty:
        raise RuntimeError(f"No rows found for model={MODEL_NAME!r} in {summary_path}")

    df["subject"] = df["subject"].map(lambda x: normalize_subject_for_cfg(x, cfg))

    out = pd.DataFrame({
        "experiment_name": exp_name,
        "dataset": cfg.dataset.dataset,
        "task": cfg.dataset.task,
        "event_preset": cfg.dataset.event_preset,
        "dataset_label": dataset_label_from_cfg(cfg),
        "subject": df["subject"],
        "lda_auc": df["auc_seed_mean"].astype(float),
        "lda_auc_seed_std": df.get("auc_seed_std", np.nan),
        "lda_pr_auc": df.get("pr_auc_seed_mean", np.nan),
        "lda_bal_acc": df.get("bal_acc_seed_mean", np.nan),
        "lda_acc": df.get("acc_seed_mean", np.nan),
        "lda_f1": df.get("f1_seed_mean", np.nan),
        "lda_summary_path": str(summary_path),
    })

    return out.reset_index(drop=True)


lda_auc_frames = []

for exp_name in EXPERIMENT_NAMES:
    lda_auc_frames.append(load_subject_level_lda_auc(exp_name))

lda_auc_df = pd.concat(lda_auc_frames, ignore_index=True)

save_dataframe(lda_auc_df, TABLE_DIR / "lda_subject_auc.csv")

display(round_numeric_columns(lda_auc_df, 4))

print("Loaded LDA subject rows:", len(lda_auc_df))
print(lda_auc_df.groupby("dataset_label")["subject"].nunique())

## ROI selection helper

For standard 10–20 datasets, the notebook uses direct centro-parietal channels such as Pz, P3, P4, CPz, Cz, CP1, and CP2.

For EGI/HydroCel-style datasets with channels named E1, E2, ..., it uses the same corrected montage-nearest logic from the ERP sanity-check notebook. This avoids the old fallback problem where the ROI became E1.

In [ ]:
def _set_montage_robust(info, montage_name: str):
    """
    Try setting a montage while remaining compatible with different MNE versions.
    """
    info = info.copy()
    montage = mne.channels.make_standard_montage(montage_name)

    try:
        info.set_montage(
            montage,
            on_missing="ignore",
            match_case=False,
            match_alias=True,
        )
    except TypeError:
        info.set_montage(
            montage,
            on_missing="ignore",
        )

    return info


def make_eeg_info(ch_names, sfreq):
    """
    Build an MNE Info object and automatically choose the montage that gives
    the largest number of valid electrode positions.

    Supports:
    - standard 10-20 names such as Fz, Cz, Pz
    - EGI/HydroCel names such as E1, E2, ..., E65/E128
    """
    ch_names = [str(ch) for ch in ch_names]

    info_base = mne.create_info(
        ch_names=ch_names,
        sfreq=float(sfreq),
        ch_types="eeg",
    )

    has_egi_names = any(re.fullmatch(r"E\d+", ch.upper()) for ch in ch_names)

    if has_egi_names:
        montage_candidates = [
            "GSN-HydroCel-65_1.0",
            "GSN-HydroCel-64_1.0",
            "GSN-HydroCel-128",
            "GSN-HydroCel-129",
            "standard_1020",
            "standard_1005",
            "biosemi64",
            "biosemi128",
        ]
    else:
        montage_candidates = [
            "standard_1020",
            "standard_1005",
            "biosemi64",
            "biosemi128",
            "GSN-HydroCel-65_1.0",
            "GSN-HydroCel-128",
        ]

    best_info = None
    best_name = None
    best_n_valid = -1

    for montage_name in montage_candidates:
        try:
            info_try = _set_montage_robust(info_base, montage_name)
            n_valid = len(valid_topomap_picks(info_try))

            if n_valid > best_n_valid:
                best_info = info_try
                best_name = montage_name
                best_n_valid = n_valid
        except Exception:
            continue

    if best_info is None:
        return info_base, "none", 0

    return best_info, best_name, best_n_valid


def _unit_normalize_positions(pos: np.ndarray) -> np.ndarray:
    """
    Normalize 3D positions to unit vectors.
    """
    pos = np.asarray(pos, dtype=float)
    norms = np.linalg.norm(pos, axis=1, keepdims=True)

    valid = np.isfinite(pos).all(axis=1) & (norms[:, 0] > 0)

    out = np.full_like(pos, np.nan, dtype=float)
    out[valid] = pos[valid] / norms[valid]

    return out


def _standard_roi_target_positions(candidates):
    """
    Get approximate standard 10-05 positions for ROI channel names.
    """
    montage = mne.channels.make_standard_montage("standard_1005")
    ch_pos = montage.get_positions()["ch_pos"]

    pos_by_upper = {
        str(name).upper(): np.asarray(pos, dtype=float)
        for name, pos in ch_pos.items()
    }

    target_positions = []
    target_names = []

    for ch in candidates:
        key = str(ch).upper()
        if key in pos_by_upper:
            target_positions.append(pos_by_upper[key])
            target_names.append(ch)

    if len(target_positions) == 0:
        for ch in ["Pz", "Cz"]:
            key = ch.upper()
            if key in pos_by_upper:
                target_positions.append(pos_by_upper[key])
                target_names.append(ch)

    if len(target_positions) == 0:
        return [], np.empty((0, 3), dtype=float)

    return target_names, np.vstack(target_positions)


def _nearest_montage_roi_indices(ch_names, candidates, sfreq=128.0, desired_k=None):
    """
    For non-standard channel names such as E1, E2, ..., choose channels whose
    montage positions are nearest to the requested centro-parietal ROI.
    """
    info, montage_name, n_valid = make_eeg_info(ch_names, sfreq)
    picks = valid_topomap_picks(info)

    if len(picks) == 0:
        return [], montage_name, None

    channel_positions = np.array(
        [info["chs"][idx]["loc"][:3] for idx in picks],
        dtype=float,
    )

    channel_positions_unit = _unit_normalize_positions(channel_positions)

    target_names, target_positions = _standard_roi_target_positions(candidates)

    if len(target_names) == 0:
        return [], montage_name, None

    target_positions_unit = _unit_normalize_positions(target_positions)

    selected = []

    for target_pos in target_positions_unit:
        if not np.isfinite(target_pos).all():
            continue

        distances = np.linalg.norm(channel_positions_unit - target_pos[None, :], axis=1)
        nearest_local = int(np.nanargmin(distances))
        nearest_global = int(picks[nearest_local])

        if nearest_global not in selected:
            selected.append(nearest_global)

    if desired_k is None:
        desired_k = min(len(candidates), len(picks))

    valid_target_positions = target_positions_unit[
        np.isfinite(target_positions_unit).all(axis=1)
    ]

    if len(valid_target_positions) > 0 and len(selected) < desired_k:
        centroid = valid_target_positions.mean(axis=0, keepdims=True)
        centroid = _unit_normalize_positions(centroid)[0]

        distances_to_centroid = np.linalg.norm(
            channel_positions_unit - centroid[None, :],
            axis=1,
        )

        sorted_local = np.argsort(distances_to_centroid)

        for local_idx in sorted_local:
            global_idx = int(picks[int(local_idx)])

            if global_idx not in selected:
                selected.append(global_idx)

            if len(selected) >= desired_k:
                break

    selected_names = [ch_names[i] for i in selected]

    return selected, montage_name, selected_names


def pick_roi_indices(ch_names, candidates, sfreq=128.0):
    """
    Robust ROI selector for susceptibility features.

    Priority:
    1. Use exact 10-20/10-05 channel names if available.
    2. Use montage-nearest channels for EGI/HydroCel-style names.
    3. Fall back to Pz/Cz if present.
    4. Last resort: channel 0.
    """
    ch_names = [str(ch) for ch in ch_names]
    name_to_idx = {ch.upper(): i for i, ch in enumerate(ch_names)}

    exact_idx = [
        name_to_idx[ch.upper()]
        for ch in candidates
        if ch.upper() in name_to_idx
    ]

    if len(exact_idx) > 0:
        return exact_idx, [ch_names[i] for i in exact_idx], "exact"

    fallback_idx = []
    for fb in ["PZ", "CZ"]:
        if fb in name_to_idx:
            fallback_idx.append(name_to_idx[fb])

    if len(fallback_idx) > 0:
        return fallback_idx, [ch_names[i] for i in fallback_idx], "fallback_exact"

    nearest_idx, montage_name, selected_names = _nearest_montage_roi_indices(
        ch_names,
        candidates,
        sfreq=sfreq,
    )

    if len(nearest_idx) > 0:
        return nearest_idx, selected_names, f"montage_nearest_{montage_name}"

    return [0], [ch_names[0]], "last_resort_first_channel"

## Feature computation

For each subject, the notebook computes interpretable subject-level features.

The main feature groups are:

- ERP separability features:
  - target-standard mean difference in the P300 window
  - absolute P300-window difference
  - P300-window RMS difference
  - ERP SNR
  - peak latency

- Noise and data-quality features:
  - baseline RMS
  - baseline variability
  - trial counts
  - class balance

- Spectral features:
  - delta, theta, alpha, beta, gamma/high-frequency power
  - low-frequency/high-frequency power ratio

These features are computed from broadly filtered epoch data, not from the already flattened LDA feature matrix.

In [ ]:
def safe_mean(x):
    x = np.asarray(x, dtype=float)
    if x.size == 0:
        return np.nan
    return float(np.nanmean(x))


def safe_std(x):
    x = np.asarray(x, dtype=float)
    if x.size == 0:
        return np.nan
    return float(np.nanstd(x))


def rms(x, eps=1e-12):
    x = np.asarray(x, dtype=float)
    if x.size == 0:
        return np.nan
    return float(np.sqrt(np.nanmean(x ** 2) + eps))


def bandpower_from_psd(freqs, psd, fmin, fmax, eps=1e-30):
    """
    Integrate PSD over a frequency band.
    """
    freqs = np.asarray(freqs, dtype=float)
    psd = np.asarray(psd, dtype=float)

    mask = (freqs >= float(fmin)) & (freqs <= float(fmax))

    if mask.sum() < 2:
        return np.nan

    return float(np.trapz(psd[mask], freqs[mask]) + eps)


def compute_subject_bandpowers(X, sfreq, bands):
    """
    Compute subject-level average spectral power across trials and channels.

    X shape: (N, C, T)
    """
    X = np.asarray(X, dtype=np.float64)

    if X.ndim != 3:
        raise ValueError(f"Expected X shape (N, C, T), got {X.shape}")

    n_times = X.shape[-1]
    nperseg = min(int(round(float(sfreq))), n_times)

    # Demean each trial/channel before PSD to reduce DC effects.
    X_psd = X - X.mean(axis=-1, keepdims=True)

    # Flatten trials and channels into one dimension:
    # (N*C, T)
    X_flat = X_psd.reshape(-1, n_times)

    freqs, psd = welch(
        X_flat,
        fs=float(sfreq),
        nperseg=nperseg,
        noverlap=nperseg // 2 if nperseg >= 8 else 0,
        axis=-1,
    )

    # Average PSD over trial-channel observations.
    psd_mean = np.nanmean(psd, axis=0)

    out = {}

    for band_name, (fmin, fmax) in bands.items():
        power = bandpower_from_psd(freqs, psd_mean, fmin, fmax)
        out[f"power_{band_name}"] = power
        out[f"log_power_{band_name}"] = float(np.log10(power + 1e-30))

    # Ratios as log-ratios are more stable than raw ratios.
    theta = out.get("power_theta_4_8", np.nan)
    alpha = out.get("power_alpha_8_12", np.nan)
    beta = out.get("power_beta_13_30", np.nan)
    low = out.get("power_low_1_12", np.nan)
    high = out.get("power_high_30_45", np.nan)
    gamma = out.get("power_gamma_30_45", np.nan)

    out["log_theta_alpha_ratio"] = float(np.log10((theta + 1e-30) / (alpha + 1e-30)))
    out["log_theta_beta_ratio"] = float(np.log10((theta + 1e-30) / (beta + 1e-30)))
    out["log_low_high_ratio"] = float(np.log10((low + 1e-30) / (high + 1e-30)))
    out["log_gamma_low_ratio"] = float(np.log10((gamma + 1e-30) / (low + 1e-30)))

    return out


def baseline_correct_epochs(X, times, baseline_window):
    """
    Baseline-correct epochs by subtracting each trial/channel's baseline mean.
    """
    X = np.asarray(X, dtype=np.float64)
    times = np.asarray(times, dtype=float)

    base_mask = (times >= baseline_window[0]) & (times <= baseline_window[1])

    if base_mask.sum() < 2:
        raise ValueError("Baseline window has too few samples.")

    baseline_mean = X[:, :, base_mask].mean(axis=-1, keepdims=True)

    return X - baseline_mean


def compute_subject_susceptibility_features(
        *,
        X,
        y,
        times,
        ch_names,
        sfreq,
        dataset,
        task,
        event_preset,
        subject,
):
    """
    Compute subject-level susceptibility features from epoch data.

    X is expected to be broad-band, minimally processed epoch data:
    shape (N, C, T)
    y: 0 = standard, 1 = target
    """
    X = np.asarray(X, dtype=np.float64)
    y = np.asarray(y).astype(int)
    times = np.asarray(times, dtype=float)
    ch_names = list(ch_names)

    if X.ndim != 3:
        raise ValueError(f"Expected X shape (N, C, T), got {X.shape}")

    if len(y) != X.shape[0]:
        raise ValueError("X/y trial count mismatch.")

    p300_mask = (times >= P300_WINDOW[0]) & (times <= P300_WINDOW[1])
    base_mask = (times >= BASELINE_WINDOW[0]) & (times <= BASELINE_WINDOW[1])
    post_mask = (times >= 0.0) & (times <= 0.6)

    if p300_mask.sum() < 2:
        raise ValueError("P300 window has too few samples.")
    if base_mask.sum() < 2:
        raise ValueError("Baseline window has too few samples.")

    roi_idx, roi_names, roi_method = pick_roi_indices(
        ch_names,
        ROI_CANDIDATES,
        sfreq=sfreq,
    )

    X_bc = baseline_correct_epochs(X, times, BASELINE_WINDOW)

    X_std = X_bc[y == 0]
    X_tgt = X_bc[y == 1]

    if len(X_std) == 0 or len(X_tgt) == 0:
        raise ValueError("Need both standard and target trials.")

    std_grand = X_std.mean(axis=0)
    tgt_grand = X_tgt.mean(axis=0)
    diff = tgt_grand - std_grand

    diff_roi = diff[roi_idx].mean(axis=0)

    p300_diff_roi = diff_roi[p300_mask]
    base_diff_roi = diff_roi[base_mask]

    p300_mean = safe_mean(p300_diff_roi)
    p300_abs_mean = abs(p300_mean)
    p300_rms = rms(p300_diff_roi)
    baseline_rms_diff = rms(base_diff_roi)

    peak_local = int(np.nanargmax(np.abs(p300_diff_roi)))
    p300_indices = np.where(p300_mask)[0]
    peak_global = int(p300_indices[peak_local])

    p300_peak_signed = float(diff_roi[peak_global])
    p300_peak_abs = abs(p300_peak_signed)
    p300_peak_latency_s = float(times[peak_global])

    erp_snr_ratio = float(p300_rms / (baseline_rms_diff + 1e-12))
    erp_snr_db = float(20.0 * np.log10(erp_snr_ratio + 1e-12))

    # Convert voltage-scale features to microvolts.
    uv = 1e6

    # Trial-level noise and variability proxies.
    baseline_epoch_values = X_bc[:, :, base_mask]
    post_epoch_values = X_bc[:, :, post_mask]
    p300_epoch_values = X_bc[:, :, p300_mask]

    roi_baseline_epoch_values = X_bc[:, roi_idx, :][:, :, base_mask]
    roi_p300_epoch_values = X_bc[:, roi_idx, :][:, :, p300_mask]

    # Single-trial ROI scores can estimate how separable target vs standard trials are
    # before any classifier is applied.
    trial_roi_p300_mean = X_bc[:, roi_idx, :][:, :, p300_mask].mean(axis=(1, 2))
    trial_roi_baseline_std = X_bc[:, roi_idx, :][:, :, base_mask].std(axis=(1, 2))

    target_trial_scores = trial_roi_p300_mean[y == 1]
    standard_trial_scores = trial_roi_p300_mean[y == 0]

    pooled_score_std = np.sqrt(
        0.5 * (
                np.nanvar(target_trial_scores)
                + np.nanvar(standard_trial_scores)
        )
    )

    trial_score_cohen_d = (
            (np.nanmean(target_trial_scores) - np.nanmean(standard_trial_scores))
            / (pooled_score_std + 1e-12)
    )

    rows = {
        "dataset": dataset,
        "task": task,
        "event_preset": event_preset,
        "dataset_label": f"{dataset}_{event_preset}" if dataset == "ds006480" else dataset,
        "subject": str(subject),

        # Trial/data-quality features
        "n_trials": int(X.shape[0]),
        "n_standard": int((y == 0).sum()),
        "n_target": int((y == 1).sum()),
        "target_rate": float((y == 1).mean()),
        "class_imbalance_abs": float(abs((y == 1).mean() - 0.5)),
        "n_channels": int(X.shape[1]),
        "n_times": int(X.shape[2]),
        "sfreq": float(sfreq),

        # ROI information
        "roi_channels": ",".join(roi_names),
        "roi_method": roi_method,
        "n_roi_channels": int(len(roi_idx)),

        # ERP separability features, microvolt scale
        "roi_p300_mean_diff_uv": float(p300_mean * uv),
        "roi_p300_abs_mean_diff_uv": float(p300_abs_mean * uv),
        "roi_p300_rms_diff_uv": float(p300_rms * uv),
        "roi_p300_peak_signed_uv": float(p300_peak_signed * uv),
        "roi_p300_peak_abs_uv": float(p300_peak_abs * uv),
        "roi_p300_peak_latency_s": p300_peak_latency_s,
        "roi_baseline_rms_diff_uv": float(baseline_rms_diff * uv),
        "roi_erp_snr_ratio": erp_snr_ratio,
        "roi_erp_snr_db": erp_snr_db,

        # Whole-head ERP difference features
        "all_ch_p300_rms_diff_uv": float(rms(diff[:, p300_mask]) * uv),
        "all_ch_baseline_rms_diff_uv": float(rms(diff[:, base_mask]) * uv),

        # Noise/variability proxies
        "baseline_rms_uv": float(rms(baseline_epoch_values) * uv),
        "poststim_rms_uv": float(rms(post_epoch_values) * uv),
        "p300_epoch_rms_uv": float(rms(p300_epoch_values) * uv),
        "baseline_std_uv": float(np.nanstd(baseline_epoch_values) * uv),
        "poststim_std_uv": float(np.nanstd(post_epoch_values) * uv),
        "roi_baseline_std_uv": float(np.nanstd(roi_baseline_epoch_values) * uv),
        "roi_p300_std_uv": float(np.nanstd(roi_p300_epoch_values) * uv),

        # Simple single-trial ERP separability proxy
        "trial_roi_p300_cohen_d": float(trial_score_cohen_d),
        "trial_roi_p300_abs_cohen_d": float(abs(trial_score_cohen_d)),
        "trial_roi_baseline_std_median_uv": float(np.nanmedian(trial_roi_baseline_std) * uv),
    }

    # Spectral power features from broad-band epoch data.
    rows.update(compute_subject_bandpowers(X, sfreq, BANDS))

    return rows

In [ ]:
feature_rows = []
failed_rows = []

for exp_name in EXPERIMENT_NAMES:
    cfg = get_experiment(exp_name)

    print("=" * 90)
    print("Feature extraction for:", exp_name)
    print("Dataset:", cfg.dataset.dataset)
    print("Task:", cfg.dataset.task)
    print("Event preset:", cfg.dataset.event_preset)
    print("=" * 90)

    # Broad raw-like feature signal.
    # This is for subject-level features, not for retraining LDA.
    feature_epoching_cfg = make_epoching_config_for_dataset(
        cfg.dataset.dataset,
        l_freq=FEATURE_L_FREQ,
        h_freq=FEATURE_H_FREQ,
        notch=FEATURE_NOTCH,
        resample_sfreq=FEATURE_RESAMPLE_SFREQ,
        tmin=EPOCH_TMIN,
        tmax=EPOCH_TMAX,
        baseline=BASELINE_WINDOW,
        event_preset=cfg.dataset.event_preset,
    )

    for subject in cfg.subjects.subjects:
        print(f"Loading {cfg.dataset.dataset} subject {subject} ...")

        try:
            subject_data, times, ch_names, sfreq = load_one_subject_grouped(
                dataset=cfg.dataset.dataset,
                task_name=cfg.dataset.task,
                subject=subject,
                epoching_cfg=feature_epoching_cfg,
                chunk_size=cfg.splits.group_chunk_size,
                apply_baseline=False,
                verbose=False,
            )

            row = compute_subject_susceptibility_features(
                X=subject_data.X,
                y=subject_data.y,
                times=times,
                ch_names=ch_names,
                sfreq=sfreq,
                dataset=cfg.dataset.dataset,
                task=cfg.dataset.task,
                event_preset=cfg.dataset.event_preset,
                subject=normalize_subject_for_cfg(subject, cfg),
            )

            row["experiment_name"] = exp_name
            feature_rows.append(row)

            print(
                f"  OK: n={row['n_trials']}, "
                f"ROI={row['roi_channels']}, "
                f"SNR={row['roi_erp_snr_db']:.2f} dB"
            )

        except Exception as e:
            print(f"  FAILED: {type(e).__name__}: {e}")

            failed_rows.append({
                "experiment_name": exp_name,
                "dataset": cfg.dataset.dataset,
                "task": cfg.dataset.task,
                "event_preset": cfg.dataset.event_preset,
                "subject": str(subject),
                "error_type": type(e).__name__,
                "error_message": str(e),
            })

feature_df = pd.DataFrame(feature_rows)
failed_df = pd.DataFrame(failed_rows)

save_dataframe(feature_df, TABLE_DIR / "subject_raw_eeg_features.csv")
save_dataframe(failed_df, TABLE_DIR / "feature_extraction_failures.csv")

print("Feature rows:", len(feature_df))
print("Failures:", len(failed_df))

display(round_numeric_columns(feature_df.head(), 4))

if len(failed_df) > 0:
    display(failed_df)

In [ ]:
subject_features_df = lda_auc_df.merge(
    feature_df,
    on=["experiment_name", "dataset", "task", "event_preset", "dataset_label", "subject"],
    how="inner",
)

missing_auc_features = lda_auc_df.merge(
    feature_df[["experiment_name", "subject"]],
    on=["experiment_name", "subject"],
    how="left",
    indicator=True,
)

missing_auc_features = missing_auc_features[
    missing_auc_features["_merge"] != "both"
    ].drop(columns=["_merge"])

save_dataframe(subject_features_df, TABLE_DIR / "subject_susceptibility_features.csv")
save_dataframe(missing_auc_features, TABLE_DIR / "subjects_missing_feature_rows.csv")

print("Merged subject-feature rows:", len(subject_features_df))
print("Rows expected from LDA AUC table:", len(lda_auc_df))

print("\nSubjects per dataset after merge:")
display(subject_features_df.groupby("dataset_label")["subject"].nunique())

if len(missing_auc_features) > 0:
    print("\nSubjects with LDA AUC but missing feature rows:")
    display(missing_auc_features)

display(round_numeric_columns(subject_features_df.head(), 4))

In [ ]:
summary_cols = [
    "lda_auc",
    "roi_p300_abs_mean_diff_uv",
    "roi_p300_rms_diff_uv",
    "roi_erp_snr_db",
    "baseline_rms_uv",
    "roi_baseline_std_uv",
    "trial_roi_p300_abs_cohen_d",
    "log_power_theta_4_8",
    "log_power_alpha_8_12",
    "log_power_beta_13_30",
    "log_power_gamma_30_45",
    "log_low_high_ratio",
    "n_trials",
    "target_rate",
]

available_summary_cols = [
    c for c in summary_cols
    if c in subject_features_df.columns
]

descriptive_summary_df = (
    subject_features_df
    .groupby("dataset_label")[available_summary_cols]
    .agg(["mean", "std", "min", "max"])
)

save_dataframe(
    descriptive_summary_df.reset_index(),
    TABLE_DIR / "descriptive_summary_by_dataset.csv",
)

display(round_numeric_columns(descriptive_summary_df, 4))

In [ ]:
ID_COLS = {
    "experiment_name",
    "dataset",
    "task",
    "event_preset",
    "dataset_label",
    "subject",
    "model",
    "roi_channels",
    "roi_method",
    "lda_summary_path",
}

TARGET_COLS = {
    "lda_auc",
    "lda_auc_seed_std",
    "lda_pr_auc",
    "lda_bal_acc",
    "lda_acc",
    "lda_f1",
}

def get_candidate_feature_columns(df: pd.DataFrame) -> list[str]:
    """
    Return numeric columns that can be correlated with LDA AUC.
    """
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    candidates = [
        c for c in numeric_cols
        if c not in ID_COLS
           and c not in TARGET_COLS
    ]

    # Exclude raw power columns by default because log-power columns are more stable.
    # Keep ratios and log powers.
    candidates = [
        c for c in candidates
        if not (c.startswith("power_") and not c.startswith("log_"))
    ]

    return candidates


def safe_corr(x, y, method="pearson"):
    """
    Safe correlation with finite filtering and constant-input handling.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    mask = np.isfinite(x) & np.isfinite(y)

    x = x[mask]
    y = y[mask]

    if len(x) < 3:
        return np.nan, np.nan, int(len(x))

    if np.nanstd(x) <= 1e-12 or np.nanstd(y) <= 1e-12:
        return np.nan, np.nan, int(len(x))

    try:
        if method == "pearson":
            r, p = pearsonr(x, y)
        elif method == "spearman":
            r, p = spearmanr(x, y)
        else:
            raise ValueError(f"Unknown method={method!r}")

        return float(r), float(p), int(len(x))

    except Exception:
        return np.nan, np.nan, int(len(x))


feature_cols = get_candidate_feature_columns(subject_features_df)

print("Candidate feature columns:", len(feature_cols))
for c in feature_cols:
    print(" -", c)

In [ ]:
within_corr_rows = []

for dataset_label, g in subject_features_df.groupby("dataset_label"):
    for feature in feature_cols:
        pearson_r, pearson_p, n_p = safe_corr(
            g["lda_auc"],
            g[feature],
            method="pearson",
        )

        spearman_r, spearman_p, n_s = safe_corr(
            g["lda_auc"],
            g[feature],
            method="spearman",
        )

        within_corr_rows.append({
            "dataset_label": dataset_label,
            "feature": feature,
            "n": n_p,
            "pearson_r": pearson_r,
            "pearson_p": pearson_p,
            "spearman_rho": spearman_r,
            "spearman_p": spearman_p,
            "abs_pearson_r": abs(pearson_r) if np.isfinite(pearson_r) else np.nan,
            "abs_spearman_rho": abs(spearman_r) if np.isfinite(spearman_r) else np.nan,
        })

within_corr_df = pd.DataFrame(within_corr_rows)

within_corr_df = within_corr_df.sort_values(
    ["dataset_label", "abs_spearman_rho"],
    ascending=[True, False],
).reset_index(drop=True)

save_dataframe(within_corr_df, TABLE_DIR / "within_dataset_correlations.csv")

display(round_numeric_columns(within_corr_df.head(40), 4))

In [ ]:
z_df = subject_features_df.copy()

z_df["lda_auc_z"] = z_df.groupby("dataset_label")["lda_auc"].transform(zscore_within_group)

for feature in feature_cols:
    z_df[f"{feature}_z"] = z_df.groupby("dataset_label")[feature].transform(zscore_within_group)

save_dataframe(z_df, TABLE_DIR / "subject_susceptibility_features_zscored.csv")

grand_corr_rows = []

for feature in feature_cols:
    feature_z = f"{feature}_z"

    pearson_r, pearson_p, n_p = safe_corr(
        z_df["lda_auc_z"],
        z_df[feature_z],
        method="pearson",
    )

    spearman_r, spearman_p, n_s = safe_corr(
        z_df["lda_auc_z"],
        z_df[feature_z],
        method="spearman",
    )

    grand_corr_rows.append({
        "feature": feature,
        "n": n_p,
        "pearson_r_z": pearson_r,
        "pearson_p_z": pearson_p,
        "spearman_rho_z": spearman_r,
        "spearman_p_z": spearman_p,
        "abs_pearson_r_z": abs(pearson_r) if np.isfinite(pearson_r) else np.nan,
        "abs_spearman_rho_z": abs(spearman_r) if np.isfinite(spearman_r) else np.nan,
    })

grand_corr_df = pd.DataFrame(grand_corr_rows)

grand_corr_df = grand_corr_df.sort_values(
    "abs_spearman_rho_z",
    ascending=False,
).reset_index(drop=True)

save_dataframe(grand_corr_df, TABLE_DIR / "zscored_grand_correlations.csv")

display(round_numeric_columns(grand_corr_df.head(40), 4))

In [ ]:
coverage_rows = []

for feature in feature_cols:
    feature_z = f"{feature}_z"

    valid = z_df[
        np.isfinite(z_df["lda_auc_z"])
        & np.isfinite(z_df[feature_z])
        ]

    coverage_rows.append({
        "feature": feature,
        "n_valid_grand": int(len(valid)),
        "n_valid_datasets": int(valid["dataset_label"].nunique()),
        "valid_datasets": ",".join(sorted(valid["dataset_label"].unique())),
    })

grand_coverage_df = pd.DataFrame(coverage_rows)

grand_corr_df = grand_corr_df.merge(
    grand_coverage_df,
    on="feature",
    how="left",
)

grand_corr_report_df = grand_corr_df[
    (grand_corr_df["n_valid_grand"] >= 40)
    & (grand_corr_df["n_valid_datasets"] >= 3)
    ].copy()

grand_corr_report_df = grand_corr_report_df.sort_values(
    "abs_spearman_rho_z",
    ascending=False,
).reset_index(drop=True)

save_dataframe(
    grand_corr_report_df,
    TABLE_DIR / "zscored_grand_correlations_report_filtered.csv",
)

display(round_numeric_columns(grand_corr_report_df.head(30), 4))

In [ ]:
# Average within-dataset correlation direction/strength for each feature.
within_summary_df = (
    within_corr_df
    .groupby("feature")
    .agg(
        mean_within_pearson_r=("pearson_r", "mean"),
        mean_abs_within_pearson_r=("abs_pearson_r", "mean"),
        mean_within_spearman_rho=("spearman_rho", "mean"),
        mean_abs_within_spearman_rho=("abs_spearman_rho", "mean"),
        n_datasets=("dataset_label", "nunique"),
    )
    .reset_index()
)

top_feature_df = grand_corr_report_df.merge(
    within_summary_df,
    on="feature",
    how="left",
)

top_feature_df = top_feature_df.sort_values(
    "abs_spearman_rho_z",
    ascending=False,
).reset_index(drop=True)

save_dataframe(top_feature_df, TABLE_DIR / "top_susceptibility_features.csv")

display(round_numeric_columns(top_feature_df.head(30), 4))

In [ ]:
top_n = 20

plot_df = grand_corr_report_df.head(top_n).copy()
plot_df = plot_df.sort_values("spearman_rho_z", ascending=True)

fig, ax = plt.subplots(figsize=(9, 7))

ax.barh(
    plot_df["feature"],
    plot_df["spearman_rho_z"],
)

ax.axvline(0.0, linewidth=1)
ax.set_xlabel("Spearman correlation with z-scored LDA AUC")
ax.set_title("Grand z-scored correlation: classification susceptibility features")
ax.grid(axis="x", alpha=0.3)

fig.tight_layout()
save_figure(fig, FIG_DIR / "grand_zscored_correlation_barplot.png")
plt.show()

In [ ]:
# Use top features from grand correlation for a compact heatmap.
heatmap_features = grand_corr_report_df.head(20)["feature"].tolist()

heat_df = within_corr_df[
    within_corr_df["feature"].isin(heatmap_features)
].copy()

heat_pivot = heat_df.pivot(
    index="feature",
    columns="dataset_label",
    values="spearman_rho",
)

# Preserve grand-ranking order.
heat_pivot = heat_pivot.loc[heatmap_features]

fig, ax = plt.subplots(figsize=(8, max(5, 0.35 * len(heatmap_features))))

im = ax.imshow(
    heat_pivot.values,
    aspect="auto",
    vmin=-1,
    vmax=1,
)

ax.set_xticks(np.arange(len(heat_pivot.columns)))
ax.set_xticklabels(heat_pivot.columns, rotation=45, ha="right")

ax.set_yticks(np.arange(len(heat_pivot.index)))
ax.set_yticklabels(heat_pivot.index)

ax.set_title("Within-dataset Spearman correlations with LDA AUC")
ax.set_xlabel("Dataset")
ax.set_ylabel("Feature")

cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Spearman rho")

fig.tight_layout()
save_figure(fig, FIG_DIR / "within_dataset_correlation_heatmap.png")
plt.show()

In [ ]:
SELECTED_FEATURES = [
    "roi_erp_snr_db",
    "roi_p300_abs_mean_diff_uv",
    "roi_p300_rms_diff_uv",
    "baseline_rms_uv",
    "roi_baseline_std_uv",
    "trial_roi_p300_abs_cohen_d",
    "log_power_theta_4_8",
    "log_power_gamma_30_45",
    "log_low_high_ratio",
    "n_trials",
]

SELECTED_FEATURES = [
    f for f in SELECTED_FEATURES
    if f in subject_features_df.columns
]

for feature in SELECTED_FEATURES:
    fig, ax = plt.subplots(figsize=(7, 5))

    for dataset_label, g in subject_features_df.groupby("dataset_label"):
        ax.scatter(
            g[feature],
            g["lda_auc"],
            label=dataset_label,
            alpha=0.8,
        )

    ax.set_xlabel(feature)
    ax.set_ylabel("Within-subject LDA ROC-AUC")
    ax.set_title(f"LDA AUC vs {feature}")
    ax.grid(True, alpha=0.3)
    ax.legend()

    fig.tight_layout()
    save_figure(fig, FIG_DIR / f"lda_auc_vs_{feature}.png")
    plt.show()

In [ ]:
top_scatter_features = grand_corr_report_df.head(6)["feature"].tolist()

for feature in top_scatter_features:
    feature_z = f"{feature}_z"

    fig, ax = plt.subplots(figsize=(7, 5))

    for dataset_label, g in z_df.groupby("dataset_label"):
        ax.scatter(
            g[feature_z],
            g["lda_auc_z"],
            label=dataset_label,
            alpha=0.8,
        )

    r_row = grand_corr_df[grand_corr_df["feature"] == feature].iloc[0]
    rho = r_row["spearman_rho_z"]
    pval = r_row["spearman_p_z"]

    ax.axhline(0.0, linewidth=1)
    ax.axvline(0.0, linewidth=1)

    ax.set_xlabel(f"{feature} z-scored within dataset")
    ax.set_ylabel("LDA AUC z-scored within dataset")
    ax.set_title(
        f"Grand z-scored susceptibility scatter\n"
        f"{feature}: Spearman rho={rho:.3f}, p={pval:.3g}"
    )
    ax.grid(True, alpha=0.3)
    ax.legend()

    fig.tight_layout()
    save_figure(fig, FIG_DIR / f"grand_z_scatter_{feature}.png")
    plt.show()

In [ ]:
# Top grand features
grand_top_for_report = grand_corr_df[[
    "feature",
    "n",
    "spearman_rho_z",
    "spearman_p_z",
    "pearson_r_z",
    "pearson_p_z",
]].head(15).copy()

save_dataframe(
    grand_top_for_report,
    TABLE_DIR / "report_top_grand_zscored_correlations.csv",
)

display(round_numeric_columns(grand_top_for_report, 4))


# Best feature per dataset
best_within_by_dataset = (
    within_corr_df
    .sort_values(["dataset_label", "abs_spearman_rho"], ascending=[True, False])
    .groupby("dataset_label")
    .head(10)
    .reset_index(drop=True)
)

save_dataframe(
    best_within_by_dataset,
    TABLE_DIR / "report_top_within_dataset_correlations.csv",
)

display(round_numeric_columns(best_within_by_dataset, 4))

In [ ]:
def direction_word(r):
    if not np.isfinite(r):
        return "unclear"
    if r > 0:
        return "positive"
    if r < 0:
        return "negative"
    return "near-zero"


print("=" * 90)
print("Classification susceptibility interpretation helper")
print("=" * 90)

print("\nTop grand z-scored correlations:")
for _, row in grand_corr_report_df.head(10).iterrows():
    feature = row["feature"]
    rho = row["spearman_rho_z"]
    p = row["spearman_p_z"]
    n = int(row["n"])

    print(
        f"- {feature}: Spearman rho={rho:.3f}, p={p:.3g}, n={n} "
        f"({strength_word(abs(rho))}, {direction_word(rho)})"
    )

print("\nPer-dataset strongest features:")
for dataset_label, g in within_corr_df.groupby("dataset_label"):
    print(f"\n{dataset_label}:")
    for _, row in g.head(5).iterrows():
        feature = row["feature"]
        rho = row["spearman_rho"]
        p = row["spearman_p"]
        n = int(row["n"])

        print(
            f"  - {feature}: rho={rho:.3f}, p={p:.3g}, n={n} "
            f"({strength_word(abs(rho))}, {direction_word(rho)})"
        )

## Interpretation notes

When reporting this analysis, avoid overinterpreting p-values because each within-dataset correlation has only approximately 13–15 subjects.

The strongest evidence should come from:

1. Features that show a large grand z-scored correlation.
2. Features that show a consistent direction across datasets.
3. Features that are physiologically interpretable, such as ERP SNR, target-standard P300-window difference, and baseline noise.

Possible interpretation patterns:

- If ERP SNR and P300-window target-standard difference correlate positively with LDA AUC:
  > Subjects with stronger target-standard ERP separability tended to be more classifiable.

- If baseline RMS/noise correlates negatively with LDA AUC:
  > Subjects with noisier baseline EEG tended to show weaker classification performance.

- If high-frequency/gamma power correlates with LDA AUC:
  > Classification susceptibility may partly reflect peripheral or muscle-related structure and should be interpreted cautiously.

- If no feature dominates:
  > Classification susceptibility appears to depend on multiple interacting subject-level factors rather than a single simple raw-data quality metric.

In [ ]:
# Top grand features using only reportable cross-dataset features.
grand_top_for_report = grand_corr_report_df[[
    "feature",
    "n",
    "n_valid_grand",
    "n_valid_datasets",
    "valid_datasets",
    "spearman_rho_z",
    "spearman_p_z",
    "pearson_r_z",
    "pearson_p_z",
]].head(15).copy()

save_dataframe(
    grand_top_for_report,
    TABLE_DIR / "report_top_grand_zscored_correlations.csv",
)

display(round_numeric_columns(grand_top_for_report, 4))


# Best feature per dataset
best_within_by_dataset = (
    within_corr_df
    .sort_values(["dataset_label", "abs_spearman_rho"], ascending=[True, False])
    .groupby("dataset_label")
    .head(10)
    .reset_index(drop=True)
)

save_dataframe(
    best_within_by_dataset,
    TABLE_DIR / "report_top_within_dataset_correlations.csv",
)

display(round_numeric_columns(best_within_by_dataset, 4))